# Coin Radar

In [ ]:
%matplotlib tk

In [ ]:
import time
import serial
import serial.tools.list_ports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from collections import deque

def _detect_arduino_port():
    hints = ("arduino", "ch340", "cp210", "usb serial", "usb-serial")
    for p in serial.tools.list_ports.comports():
        blob = f"{p.device} {p.description} {p.manufacturer}".lower()
        if any(h in blob for h in hints):
            return p.device
    ports = list(serial.tools.list_ports.comports())
    return ports[0].device if ports else "COM3"

SERIAL_PORT = _detect_arduino_port()
print("Puerto serial:", SERIAL_PORT)
BAUD_RATE = 9600
MAX_RANGE_CM = 100
MIN_DETECT_CM = 2
OUT_OF_RANGE_CM = MAX_RANGE_CM + 1
# Debe coincidir con coin_radar_100cm.ino
ANGLE_MIN_DEG = 30
ANGLE_MAX_DEG = 150
ANGLE_OFFSET_DEG = 0
RING_STEP_CM = 25
FADE_SECONDS = 3.0
SWEEP_TRAIL = 15

In [ ]:
try:
    ser.close()          # por si quedó abierto de una corrida anterior
except NameError:
    pass

ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=0)
time.sleep(2.5)
ser.reset_input_buffer()

_buffer = ""

def _try_reconnect():
    """Intenta reabrir el puerto tras un error de driver/USB (ClearCommError, etc.)
    en vez de dejar que la animación se detenga por completo."""
    global ser
    try:
        ser.close()
    except Exception:
        pass
    time.sleep(0.3)
    try:
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=0)
    except serial.SerialException:
        pass  # seguimos sin puerto; se reintenta en el próximo frame

def read_all_readings():
    """Lee TODO lo pendiente en el puerto y devuelve lista de (angulo, distancia).
    Si el driver falla (típico en Windows: 'ClearCommError'/'Controlador no válido'),
    reintenta reconectar en vez de lanzar la excepción y tumbar la animación."""
    global _buffer
    try:
        n = ser.in_waiting
        if n:
            _buffer += ser.read(n).decode(errors="ignore")
    except serial.SerialException:
        status.set_text("Puerto desconectado, reconectando...")
        _try_reconnect()
        return []
    readings = []
    while "." in _buffer:
        msg, _, _buffer = _buffer.partition(".")
        a, sep, d = msg.partition(",")
        if not sep:
            continue
        try:
            readings.append((int(a), int(d)))
        except ValueError:
            continue 
    return readings

def read_latest_reading():
    """Una sola lectura por frame = misma posición que el servo en ese instante."""
    batch = read_all_readings()
    return batch[-1] if batch else None

In [ ]:
plt.close("all")
plt.style.use("dark_background")
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection="polar")
ax.set_theta_zero_location("E")
ax.set_theta_direction(1)
ax.set_thetamin(ANGLE_MIN_DEG)
ax.set_thetamax(ANGLE_MAX_DEG)
ax.set_ylim(0, MAX_RANGE_CM)
ax.set_rticks(np.arange(RING_STEP_CM, MAX_RANGE_CM + 1, RING_STEP_CM))
ax.set_rlabel_position(0)
ax.tick_params(colors="lime")
ax.grid(color="lime", alpha=0.4)

trail_lines = [ax.plot([], [], color="lime", linewidth=3, alpha=0)[0] for _ in range(SWEEP_TRAIL)]
recent_angles = deque(maxlen=SWEEP_TRAIL)
sweep_line, = ax.plot([], [], color="#ff4444", linewidth=2.5, alpha=0.9)

detections = []
detection_artist = None

status = fig.text(0.5, 0.93, "Esperando datos...", ha="center", fontsize=14, color="lime")

last_reading_time = None
STALE_TIMEOUT_S = 0.25  # si no llega un dato nuevo en este tiempo, se muestra "Indeterminado"

def in_range(dist):
    return dist is not None and MIN_DETECT_CM <= dist <= MAX_RANGE_CM

def update(_):
    global detection_artist, last_reading_time
    now = time.time()

    batch = read_all_readings()
    for angle, dist in batch:
        if in_range(dist):
            detections.append((np.radians(angle + ANGLE_OFFSET_DEG), dist, now))

    if batch:
        last_reading_time = now
        angle, dist = batch[-1]
        angle_disp = angle + ANGLE_OFFSET_DEG
        theta = np.radians(angle_disp)
        recent_angles.append(theta)
        if dist == -1:
            # El Arduino no recibió ningún eco válido: estado indeterminado explícito,
            # no es "muy lejos", es que no se pudo medir nada esta vez.
            sweep_line.set_data([theta, theta], [0, MAX_RANGE_CM])
            status.set_text(f"Servo {angle}°   Indeterminado (sin eco)")
        elif in_range(dist):
            sweep_line.set_data([theta, theta], [0, dist])
            status.set_text(f"Servo {angle}°   {dist} cm   DETECTADO")
        else:
            sweep_line.set_data([theta, theta], [0, MAX_RANGE_CM])
            status.set_text(f"Servo {angle}°   fuera de rango ({dist} cm)")
    elif last_reading_time is not None and (now - last_reading_time) > STALE_TIMEOUT_S:
        # No ha llegado ningún dato nuevo en un rato: no dejamos el texto viejo
        # congelado como si siguiera vigente.
        status.set_text("Indeterminado (esperando dato nuevo...)")

    angles = list(recent_angles)
    for i, line in enumerate(trail_lines):
        k = len(angles) - 1 - i
        if k >= 0:
            line.set_data([angles[k]] * 2, [0, MAX_RANGE_CM])
            line.set_alpha(max(0.0, 1.0 - i / SWEEP_TRAIL))
        else:
            line.set_alpha(0)

    detections[:] = [d for d in detections if now - d[2] < FADE_SECONDS]
    if detection_artist is not None:
        detection_artist.remove()
        detection_artist = None

    if detections:
        th = np.array([d[0] for d in detections])
        di = np.array([d[1] for d in detections])
        al = np.array([1.0 - (now - d[2]) / FADE_SECONDS for d in detections])
        colors = np.column_stack([
            np.ones(len(al)),
            np.full(len(al), 0.12),
            np.full(len(al), 0.12),
            np.clip(al, 0.15, 1.0),
        ])
        detection_artist = ax.scatter(
            th,
            di,
            s=180,
            c=colors,
            edgecolors="white",
            linewidths=0.6,
            zorder=5,
        )
    return []

ani = animation.FuncAnimation(fig, update, interval=30, blit=False, cache_frame_data=False)
plt.show()

In [ ]:
#ser.close()

Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\CARLOS\AppData\Local\Programs\Python\Python310\lib\tkinter\__init__.py", line 1921, in __call__
    return self.func(*args)
  File "c:\Users\CARLOS\AppData\Local\Programs\Python\Python310\lib\tkinter\__init__.py", line 839, in callit
    func(*args)
  File "c:\Users\CARLOS\AppData\Local\Programs\Python\Python310\lib\site-packages\matplotlib\backends\_backend_tk.py", line 142, in _on_timer
    super()._on_timer()
  File "c:\Users\CARLOS\AppData\Local\Programs\Python\Python310\lib\site-packages\matplotlib\backend_bases.py", line 1226, in _on_timer
    ret = func(*args, **kwargs)
  File "c:\Users\CARLOS\AppData\Local\Programs\Python\Python310\lib\site-packages\matplotlib\animation.py", line 1426, in _step
    still_going = super()._step(*args)
  File "c:\Users\CARLOS\AppData\Local\Programs\Python\Python310\lib\site-packages\matplotlib\animation.py", line 1119, in _step
    self._draw_next_frame(framedata, se